# ML model analyse

This notebook is used to load the model and analyse the results

In [4]:
import pandas as pd
import torch 
import numpy as np
import torch.nn as nn
from torchsummary import summary
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from datetime import datetime
import neptune
import model as mf

from torchvision.io import read_image
from torchvision.transforms.functional import to_tensor


DATA_PATH = "dataset/" #Do not change folder structure that was created by Kasper
# dataset/
#   DRR_images/
#       [n].png
#   excel/
#       *.xlsx 


In [5]:
labels = pd.read_excel(DATA_PATH+"excel/all_DRR_scores.xlsx")
labels['fullpath'] = DATA_PATH+"DRR_images/"+labels['DRR filename']+".png"
labels['score_classified'] = pd.qcut(labels['score'], 5).factorize()[0]

#labels.iloc[0]['fullpath']
labels

,DRR filename,aneurysm score,neck score,score,fullpath,score_classified
0,C0001_X0_Y0,12.272727,1.783784,14.056511,dataset/DRR_images/C0001_X0_Y0.png,0
1,C0001_X10_Y0,7.358974,1.584906,8.943880,dataset/DRR_images/C0001_X10_Y0.png,1
2,C0001_X20_Y0,5.538462,1.921569,7.460030,dataset/DRR_images/C0001_X20_Y0.png,1
3,C0001_X30_Y0,5.309091,2.074074,7.383165,dataset/DRR_images/C0001_X30_Y0.png,1
4,C0001_X40_Y0,4.949153,1.689189,6.638342,dataset/DRR_images/C0001_X40_Y0.png,2
...,...,...,...,...,...,...
6835,C0074_X310_Y180,2.008721,2.130435,4.139156,dataset/DRR_images/C0074_X310_Y180.png,3
6836,C0074_X320_Y180,1.817175,1.709091,3.526265,dataset/DRR_images/C0074_X320_Y180.png,4
6837,C0074_X330_Y180,1.527578,2.289474,3.817052,dataset/DRR_images/C0074_X330_Y180.png,4
6838,C0074_X340_Y180,3.066038,2.050000,5.116038,dataset/DRR_images/C0074_X340_Y180.png,3


In [10]:
import torch.utils.data.dataloader


device = torch.device("cuda")
model = mf.Block()
data_loader = torch.utils.data.DataLoader(mf.CustomImageDataset(labels), batch_size=64, shuffle=True)

In [1]:
train, labels = next(iter(data_loader))
print(f"Feature batch shape: {train.size()}")
print(f"Feature batch type: {train.type()}")
print(f"Labels batch shape: {labels.size()}")
img = train[31].squeeze()
label = labels[0]
plt.imshow(img, cmap="gray")
plt.show()
print(f"Label: {label}")

NameError: name 'data_loader' is not defined

In [48]:
model.load_state_dict(torch.load("mymodel_classified.pth"))
model.to(device)
outputs = model(train.to(device))

In [49]:
labels

tensor([4, 0, 0, 4, 1, 0, 1, 0, 2, 4, 2, 0, 2, 2, 0, 0, 0, 1, 3, 4, 2, 4, 0, 0,
        4, 0, 1, 2, 2, 1, 1, 3, 2, 0, 0, 2, 0, 2, 2, 1, 3, 1, 3, 2, 3, 4, 0, 1,
        4, 3, 4, 4, 3, 4, 1, 1, 4, 4, 4, 2, 1, 2, 4, 0])

In [51]:
_, predicted = torch.max(outputs,1)
total = len(labels)
correct = 0

for label, prediction in zip(labels, predicted.detach().cpu()):
    if label == prediction:
        #print(label, prediction)
        correct = correct + 1

print(f'Accuracy: {(correct/total):5f}')

Accuracy: 0.437500
